# Start the extracted CTI RAG API in Google Colab

This launcher runs the extracted RAG engine (copied from the original notebook) and exposes a temporary HTTPS API URL for the Vercel frontend.

**Prerequisite:** Clone or upload the same `CTI-RAG-Assistant` repository into `/content/CTI-RAG-Assistant`. The original assessment notebook remains unchanged.

In [ ]:
# 1. Set the backend working directory.
%cd /content/CTI-RAG-Assistant/backend

In [ ]:
# 2. Install backend dependencies.
# Colab already provides CUDA-enabled PyTorch. Avoid reinstalling torch here.
%pip install -q fastapi "uvicorn[standard]" datasets pandas numpy requests transformers accelerate bitsandbytes sentence-transformers chromadb langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters rank-bm25

In [ ]:
# 3. Load secrets into this Colab runtime (never put them in GitHub or Vercel).
import os
from google.colab import userdata

for secret_name in ("HF_TOKEN", "THREATCLUSTER_API_KEY"):
    try:
        value = userdata.get(secret_name)
        if value:
            os.environ[secret_name] = value
    except Exception:
        pass

if not os.getenv("HF_TOKEN") and not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    raise RuntimeError("Add HF_TOKEN under Colab > Secrets, then rerun this cell.")

print("Secrets configured (values are not displayed).")

In [ ]:
# 4. Import the extracted engine and API.
# This loads the same datasets, Chroma/BM25 indexes, Gemma 2 2B NF4 model,
# prompts, retrieval settings, ThreatCluster connector and email rules.
import sys
sys.path.insert(0, "/content/CTI-RAG-Assistant/backend")

from api import app
print("CTI RAG API and model initialisation completed.")

In [ ]:
# 5. Start FastAPI inside this Colab runtime and create a temporary HTTPS tunnel.
import subprocess, threading, time, re, urllib.request, os, stat
import uvicorn

server_config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(server_config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

# Wait for local API readiness.
import requests
for attempt in range(90):
    try:
        health = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if health.ok:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("FastAPI did not become ready. Check the cell output.")

# Download Cloudflare's temporary tunnel client.
cloudflared_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared_path):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_path
    )
    os.chmod(cloudflared_path, 0o755)

tunnel = subprocess.Popen(
    [cloudflared_path, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

colab_api_url = None
deadline = time.time() + 90
while time.time() < deadline and colab_api_url is None:
    line = tunnel.stdout.readline()
    print(line.rstrip())
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        colab_api_url = match.group(0)

if not colab_api_url:
    raise RuntimeError("Could not obtain the temporary HTTPS URL. Inspect the tunnel output.")

print("\nCTI API URL:", colab_api_url)
print("Health check:", colab_api_url + "/health")
print("Keep this Colab session running while using the Vercel frontend.")